# Used Car Prediction Model
**Beginner-friendly regression project — CampusX learning scope through approximately Day 40.**

Predict used-car prices from the supplied Kaggle CSV. This notebook practices EDA, feature engineering, imputation, one-hot encoding, `ColumnTransformer`, `Pipeline`, train/test split, Linear Regression, Random Forest, and evaluation. Target column: `price_usd`.

## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datetime import date
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
pd.set_option('display.max_columns', None)
sns.set_theme()

## 2. Upload and load the CSV

In [ ]:
uploaded = files.upload()
csv_files = [name for name in uploaded if name.lower().endswith('.csv')]
if not csv_files:
    raise ValueError('Upload the Kaggle CSV file.')
df = pd.read_csv(csv_files[0])
print('Shape:', df.shape)
display(df.head())

## 3. Inspect and clean basic issues

In [ ]:
print('Columns:', df.columns.tolist())
df.info()
display(df.isna().sum().sort_values(ascending=False).to_frame('missing_values'))
print('Duplicate rows:', df.duplicated().sum())
display(df.describe(include='all').T)

if 'price_usd' not in df.columns:
    raise ValueError("Expected target column 'price_usd' was not found.")
df = df.drop_duplicates().dropna(subset=['price_usd']).copy()
df['price_usd'] = pd.to_numeric(df['price_usd'], errors='coerce')
df = df.dropna(subset=['price_usd']).copy()

## 4. Exploratory data analysis (EDA)

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df['price_usd'], kde=True)
plt.title('Distribution of Used Car Prices')
plt.xlabel('Price (USD)')
plt.show()

numeric_df = df.select_dtypes(include=np.number)
plt.figure(figsize=(9, 6))
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Between Numerical Features')
plt.show()

if 'brand' in df.columns:
    plt.figure(figsize=(12, 5))
    sns.boxplot(data=df, x='brand', y='price_usd')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 5. Feature engineering: car age
We derive `car_age` from `make_year` and drop the original year column to avoid using both versions of the same information.

In [ ]:
if 'make_year' in df.columns:
    df['car_age'] = date.today().year - pd.to_numeric(df['make_year'], errors='coerce')
    df = df.drop(columns=['make_year'])
display(df.head())

## 6. Split features and target, then train/test split

In [ ]:
X = df.drop(columns=['price_usd']).copy()
y = df['price_usd'].copy()
numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()
print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print('Train rows:', len(X_train), '| Test rows:', len(X_test))

## 7. Preprocessing with `ColumnTransformer` and `Pipeline`
Numerical values use median imputation and scaling. Categorical values use most-frequent imputation and one-hot encoding. Fitting preprocessing inside the pipeline helps prevent test-set data leakage.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

## 8. Train and compare two regression models

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(
        n_estimators=150, random_state=RANDOM_STATE,
        min_samples_leaf=2, n_jobs=-1
    )
}

trained_pipelines = {}
results = []
for name, model in models.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    results.append({
        'Model': name,
        'MAE': mean_absolute_error(y_test, pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, pred)),
        'R2': r2_score(y_test, pred)
    })
    trained_pipelines[name] = pipe

results_df = pd.DataFrame(results).sort_values('MAE')
display(results_df)

## 9. Evaluate the best model by MAE
- **MAE:** average absolute error in the target's units (USD here).
- **RMSE:** penalizes large errors more than MAE.
- **R²:** compares the model with predicting the test-set mean; it can be negative.

Metrics describe this dataset and split; they are not a guarantee of real-market performance.

In [ ]:
best_model_name = results_df.iloc[0]['Model']
best_pipeline = trained_pipelines[best_model_name]
test_predictions = best_pipeline.predict(X_test)
print('Best model by test MAE:', best_model_name)
print('MAE: $', round(mean_absolute_error(y_test, test_predictions), 2))
print('RMSE: $', round(np.sqrt(mean_squared_error(y_test, test_predictions)), 2))
print('R²:', round(r2_score(y_test, test_predictions), 4))

plt.figure(figsize=(7, 6))
sns.scatterplot(x=y_test, y=test_predictions, alpha=0.6)
low = min(float(y_test.min()), float(np.min(test_predictions)))
high = max(float(y_test.max()), float(np.max(test_predictions)))
plt.plot([low, high], [low, high], linestyle='--')
plt.xlabel('Actual price (USD)')
plt.ylabel('Predicted price (USD)')
plt.title(f'Actual vs Predicted — {best_model_name}')
plt.tight_layout()
plt.show()

## 10. Example prediction and save model

In [ ]:
sample_car = X_test.iloc[[0]].copy()
sample_prediction = best_pipeline.predict(sample_car)[0]
display(sample_car)
print(f'Predicted price: ${sample_prediction:,.2f} USD')

import joblib
joblib.dump(best_pipeline, 'used_car_price_pipeline.joblib')
print('Saved pipeline to used_car_price_pipeline.joblib')

## Conclusion and limitations
This is a learning project, not a production valuation tool. Dataset currency, market, and data quality affect performance. If the dataset is synthetic, metrics may reflect the way it was generated rather than real used-car markets. The pipeline keeps preprocessing and model steps together for consistent predictions.